# Lab 09: Assignment 2 Tutorial

In Assignment 1, we followed a wave packet in a harmonic well. Now put it in the right-hand well of a double-well potential.

A classical particle below the barrier cannot cross. What will the quantum probability density do, and how can we tell when it has reached the left well?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

# if you're curious, this is to change default formatting  
# make plots look good in the slideshow format.
plt.rcParams.update({"font.size": 13,
    "figure.figsize": (6.2, 3.3), "figure.dpi": 100,
    "figure.constrained_layout.use": True})

<details class="lab09-answer">
<summary>Answer and explanation</summary>

The packet can tunnel: probability can move to the left well even though a classical particle with energy below the barrier cannot cross. Follow the density, not just its mean. A negative mean alone does not show that most of the probability has transferred.

</details>

## Put the wavefunction on a grid

We will store one complex amplitude at each position. Use $m=\hbar=1$ and 600 intervals on $[-4,4]$.

In [ ]:
# coding culture: we tend to use capital letters when want to define 
# immutable constants that will be used throughout a whole script.
ALPHA = 0.4
HBAR = 1
BOXSIZE = 8
ND = 600
DELTAX = BOXSIZE / ND
XMIN = 1 / np.sqrt(ALPHA)
x = np.linspace(-BOXSIZE / 2, BOXSIZE / 2, ND + 1)
print("Positions:", x.size, "Spacing:", DELTAX)

There are 601 positions and 600 intervals. The spacing $\Delta x$ will weight our spatial sums.

<details class="lab09-answer">
<summary>About the constants</summary>

Capitals are a naming convention, not a Python restriction: `ALPHA` can still be reassigned. If you change it, rerun the setup and the later calculations so the well centers, initial state, and propagator use the same value.

</details>

## Two wells and a barrier

$$
V(x)=\alpha x^4-2x^2+\frac{1}{\alpha}.
$$

We'll use $\alpha=0.4$ here. Use your assigned value in the homework.

In [ ]:
# and then we use lowercase for more generic variables.
@njit
def V(x, alpha):
    return alpha * x**4 - 2 * x**2 + 1 / alpha

alpha_view = ALPHA
fig, ax = plt.subplots()
ax.plot(x, V(x, alpha_view), color="black")
ax.set(xlabel="x", ylabel="V(x)",
       xlim=(-4, 4), ylim=(0, 6),
       title=f"alpha = {alpha_view}")
ax.grid(True) 
plt.show()

Raise `alpha_view` to 0.6. How do the barrier and separation change? This changes only the preview; the evolution uses `ALPHA`.

<details class="lab09-answer">
<summary>Answer and explanation</summary>

The barrier gets lower and the wells move closer together. From $\alpha=0.4$ to 0.6, the height falls from 2.5 to about 1.67, and the separation $2/\sqrt{\alpha}$ falls from about 3.16 to 2.58. Editing `alpha_view` changes this preview; it does not change the evolution.

</details>

## After changing alpha

In [ ]:
# and then we use lowercase for more generic variables
@njit
def V(x, alpha):
    return alpha * x**4 - 2 * x**2 + 1 / alpha

alpha_view = .6 # <==========================================
fig, ax = plt.subplots()
ax.plot(x, V(x, alpha_view), color="black")
ax.set(xlabel="x", ylabel="V(x)",
       xlim=(-4, 4), ylim=(0, 6),
       title=f"alpha = {alpha_view}")
ax.grid(True) 
plt.show()

This preview uses $\alpha=0.6$. `ALPHA` is still 0.4.

## Near each minimum

The potential is locally quadratic (first order taylor expansion):

$$
V_\pm(x)=4(x\mp x_{\min})^2,
$$

with $x_{\min}=1/\sqrt{\alpha}$

In [ ]:
alpha_view = ALPHA # let's not forget to turn this back to our universal CONSTANT

def V_plus(x, alpha): 
    return 4 * (x - 1 / np.sqrt(alpha))**2
def V_minus(x, alpha):
    return 4 * (x + 1 / np.sqrt(alpha))**2

def plot_potentials(ax, v_color, posv_color, negv_color):
    ax.plot(x, V(x, ALPHA), color=v_color)
    ax.plot(x, V_plus(x, ALPHA), "--", color=posv_color)
    ax.plot(x, V_minus(x, ALPHA), ":", color=negv_color)
    ax.set(ylabel="Potential", ylim=(0, 6),
           zorder=0)
    ax.legend(["V", "V+", "V-"],
              loc="upper right", fontsize=10)
    
x_window = (-4, 4)
fig, ax = plt.subplots()

plot_potentials(ax, v_color = ('black', 0.15), posv_color='red', negv_color='blue') # can you guess how to colors work here?
ax.set(xlabel="x", xlim=x_window)
plt.show()

Where does the approximation fail?

<details class="lab09-answer">
<summary>Answer and explanation</summary>

Each parabola agrees best near its own minimum. Moving toward the barrier or far outside that well, the neglected cubic and quartic terms matter. In particular, the parabola centered in one well cannot describe the other minimum.

The quadratic term is the *second-order* Taylor approximation. At either minimum, $V=0$, $V^{\prime}=0$, and $V^{\prime\prime}=8$, so $V(x_{\min}+y)\approx\tfrac12(8)y^2=4y^2$. A first-order expansion at the minimum would be flat.

`V+` is red and centered on the right well; `V-` is blue and centered on the left. In `("black", 0.15)`, 0.15 is the color opacity: 0 is transparent and 1 is opaque. This color alpha is separate from the physical parameter `ALPHA`.

</details>

## The initial wavefunction

We're approximating both wells locally as harmonic, so we might as well approximate the state as a ground state oscillator, with $\omega=2\sqrt{2}$:

$$
\psi(x)=\left(\frac{\omega}{\pi}\right)^{1/4}
\exp\left[-\frac{\omega}{2}(x-x_c)^2\right],
$$

which remember is just a gaussian.

In [ ]:
OMEGA = np.sqrt(8)
def func_psi_0(x, x_start):
    return (OMEGA / np.pi)**0.25 * np.exp(
        -OMEGA * (x - x_start)**2 / (2 * HBAR))

psi_0 = func_psi_0(x, XMIN)
center = XMIN

fig, ax = plt.subplots()
ax.plot(x, func_psi_0(x, center), color='red')
ax.set(xlabel="x", ylabel="Wavefunction",
       zorder=1, xlim=(-4, 4), ylim=(0, 1.1),
       title=f"Center = {center:.2f}")

ax.patch.set_visible(False)
plot_potentials(ax.twinx(), v_color = ('black', .05), posv_color=('red', .15), negv_color=('blue', .1))
plt.show()

<details class="lab09-answer">
<summary>Why this Gaussian?</summary>

Compare $4(x-x_c)^2$ with $\tfrac12m\omega^2(x-x_c)^2$. With $m=1$, this gives $\omega^2=8$. The Gaussian is the ground state of that local oscillator; it is not an exact energy eigenstate of the full double well. Read its amplitude on the left axis and the potentials on the right.

</details>

## Why can a localized state move?

The full potential has even and odd energy eigenstates (state functions). 
Our right-well Gaussian is neither.
So to actually allow it to physically make sense globally, 
we need to construct even and odd versions of the state.

$$
\psi_{\mathrm S}=\frac{\psi_++\psi_-}{\sqrt{2}},\qquad
\psi_{\mathrm A}=\frac{\psi_+-\psi_-}{\sqrt{2}}.
$$

### Even/odd function refresher
<details class="lab09-answer">
<summary>Show the reflection</summary>

Reflecting $x$ swaps the two local Gaussians: $\psi_+(-x)=\psi_-(x)$ and $\psi_-(-x)=\psi_+(x)$. Therefore

$$
\begin{aligned}
\psi_{\mathrm S}(-x)&=\frac{\psi_-(x)+\psi_+(x)}{\sqrt2}=\psi_{\mathrm S}(x),\\
\psi_{\mathrm A}(-x)&=\frac{\psi_-(x)-\psi_+(x)}{\sqrt2}=-\psi_{\mathrm A}(x).
\end{aligned}
$$

The even combination stays the same; the odd combination changes sign. **Even** does not mean positive everywhere.

</details>

In [ ]:
psi_plus = func_psi_0(x, XMIN)
psi_minus = func_psi_0(x, -XMIN)
psi_S = (psi_plus + psi_minus) / np.sqrt(2)
psi_A = (psi_plus - psi_minus) / np.sqrt(2)

fig, ax = plt.subplots()
ax.plot(x, psi_S, label="Symmetric", color='lightgreen')
ax.plot(x, psi_A, label="Antisymmetric", color='magenta')
ax.axhline(0, color="gray", linewidth=0.7)
ax.set(xlabel="x", ylabel="Wavefunction",
       zorder=1, xlim=(-4, 4), ylim=(-1, 1.5))
ax.legend(loc="upper left", fontsize=10)
ax.patch.set_visible(False)
plot_potentials(ax.twinx(), v_color = ('black', .05), posv_color=('red', .15), negv_color=('blue', .1))
plt.show()

These approximate the lowest even and odd states. Different phase rates allow their superposition to change its density. Which combination has a node at the barrier?

<details class="lab09-answer">
<summary>Answer and explanation</summary>

The antisymmetric combination has the node: $\psi_{\mathrm A}(0)=-\psi_{\mathrm A}(0)$, so it must be zero there. The symmetric combination is even and adds the two positive Gaussian amplitudes at the origin. Even means unchanged by reflection, not necessarily positive everywhere.

The localized Gaussian is already a valid quantum state. It does not have to be even or odd. Symmetry makes parity a useful way to describe the energy eigenstates; it is not a requirement on every state.

The definitions give $\psi_+=(\psi_{\mathrm S}+\psi_{\mathrm A})/\sqrt2$. Exact eigenstates with different energies acquire different phases, so their interference can change the density. These Gaussian combinations are approximations, and their overlap means they are not exactly normalized by the factor $1/\sqrt2$.

</details>

## Advance the amplitudes

$$
\tilde{\mathcal K}(x_b,\Delta t;x_a,0)=
\exp\left[i\left(\frac{(x_b-x_a)^2}{2\Delta t}
-V\left(\frac{x_a+x_b}{2}\right)\Delta t\right)\right].
$$

In [ ]:
@njit
def func_K(x_a, x_b, dt, alpha):
    kinetic = 0.5 * (x_b - x_a)**2 / dt
    potential = V((x_a + x_b) / 2, alpha) * dt
    return np.exp(1j * (kinetic - potential))

<details class="lab09-answer">
<summary>About the propagator</summary>

The exponent contains the short-step kinetic action minus the midpoint potential action. The kernel is a complex amplitude, so its real part is not a probability. This `func_K` omits the common scalar prefactor; the later step function normalizes the updated state.

</details>

## From the propagator to the next state

Add the complex amplitudes from every starting position. On the grid, the integral becomes a weighted sum:

$$
\begin{aligned}
\psi(x_b,t+\Delta t)&\approx\int \mathcal K(x_b,x_a;\Delta t)\psi(x_a,t)\,dx_a,\\
\psi_i(t+\Delta t)&\approx\Delta x\sum_j K_{ij}\psi_j(t).
\end{aligned}
$$

Why do we add the amplitudes before taking the absolute square?

<details class="lab09-answer">
<summary>Answer and explanation</summary>

The alternatives interfere. For two contributions, $ |a+b|^2=|a|^2+|b|^2+2\operatorname{Re}(a^*b)$. Squaring each contribution before adding would remove that interference term. First sum the complex amplitudes; then take the absolute square.

</details>

## Build the matrix

Each column starts at one position; each row arrives at one position. Use the assignment's $\Delta t=\pi/128$ for the evolution.

Compile the loops with `@njit`, as in Lab 08. The first call includes compilation.

In [ ]:
@njit
def build_kernel(x, dt, alpha):
    kernel = np.zeros((x.size, x.size),
                      dtype=np.complex64)
    for i in range(x.size):
        for j in range(x.size):
            kernel[i, j] = func_K(
                x[j], x[i], dt, alpha)
    return kernel.astype(np.complex128)

DELTAT = np.pi / 128
K_dt = build_kernel(x, DELTAT, ALPHA)
print("Kernel shape:", K_dt.shape)

For a fixed starting position $j$, which slice gives all of its destination amplitudes?

<details class="lab09-answer">
<summary>Answer and explanation</summary>

`K_dt[:, j]`: keep the starting-position column fixed and take every destination row. By contrast, `K_dt[i, :]` collects all starting positions that contribute to one destination.

The kernel is still calculated and rounded to `complex64`, then stored as `complex128`. This keeps the same entries while matching the state dtype for Numba's matrix multiplication.

</details>

## Take a step and inspect it

The grid sum is a matrix–vector product. Normalize because our kernel omits the scalar prefactor.

In [ ]:
@njit
def step(state, kernel, dx):
    updated = dx * (kernel @ state)
    norm = dx * np.sum(np.abs(updated)**2)
    return updated / np.sqrt(norm)
    
steps_view = 20
psi_view = psi_0.astype(np.complex128)
for _ in range(steps_view):
    psi_view = step(psi_view, K_dt, DELTAX)
    
fig, ax = plt.subplots()
ax.plot(x, np.abs(psi_0)**2, "--", label="t = 0")
ax.plot(x, np.abs(psi_view)**2, label="Later")
ax.set(xlabel="x", ylabel="Probability density",
       zorder=1, xlim=(-4, 4), ylim=(0, 1.3))
ax.legend(loc="upper left", fontsize=10)
ax.patch.set_visible(False)

plot_potentials(ax.twinx(), v_color = ('black', .5), posv_color=('red', 0), negv_color=('blue', 0))
plt.show()

Try `steps_view = 100`. Does the shape change? Normalization keeps total probability at one; it does not establish accuracy.

<details class="lab09-answer">
<summary>Answer and explanation</summary>

The packet changes shape; it is not simply a rigid Gaussian sliding across the plot. The starting Gaussian is adapted to the local parabola, while the evolution uses the full anharmonic potential. A norm of 1 checks the normalization step, not the accuracy of the propagated motion.

The unnormalized update is `dx * (kernel @ state)`. Compute its grid norm $N=\Delta x\sum_i|\psi_i|^2$, then divide the amplitudes by $\sqrt N$. Dividing by $N$ itself would not produce norm 1.

The first call to `step` compiles it; later calls reuse the compiled function. Rerunning the cell containing `@njit` creates a new function, so that also starts compilation again.

</details>

## Follow the motion

Run to $T=20\pi$. The mean is the probability-weighted average of position:

$$
\langle x\rangle(t)\approx\Delta x\sum_i x_i|\psi(x_i,t)|^2.
$$

In [ ]:
T0 = 20 * np.pi
NT = int(T0 / DELTAT)
t = np.arange(NT + 1) * DELTAT
psi = [psi_0.astype(np.complex128)]
for _ in range(NT):
    psi.append(step(psi[-1], K_dt, DELTAX))
prob = np.abs(np.asarray(psi))**2
x_bar = DELTAX * (prob @ x)

fig, ax = plt.subplots()
ax.plot(t, x_bar)
ax.axhline(0, color="gray", linewidth=0.7)
ax.set(xlabel="Time", ylabel="Mean position",
       xlim=(0, t[-1]))
plt.show()

Find the first broad minimum, rather than one of the faster fluctuations. Has most of the probability reached the left well?

<details class="lab09-answer">
<summary>Answer and explanation</summary>

For this $\alpha=0.4$ example, the first broad minimum is near $t=28.15$, with $\langle x\rangle\approx-1.51$. About 98% of the probability is at $x<0$. The small ripples give several nearby local minima; read the broad transfer first.

Use the first broad trough of the slow motion, not the first small ripple or a minimum from a later cycle. Then inspect the density at that time. To check whether most probability is on the left, use the grid sum over $x<0$; a negative mean alone is insufficient.

</details>

## Check what the mean is telling us

In [ ]:
time_view = 0.0
n = np.argmin(np.abs(t - time_view))
fig, ax = plt.subplots()
ax.plot(x, prob[n])
ax.set(xlabel="x", ylabel="Probability density",
       zorder=1, xlim=(-4, 4),
       ylim=(0, 1.1 * prob.max()),
       title=f"t = {t[n]:.3f}")
ax.patch.set_visible(False)
plot_potentials(ax.twinx(), v_color = ('black', .5), posv_color=('red', 0), negv_color=('blue', 0))
plt.show()

Set `time_view` near that minimum. Is the probability transferred or split? Would $\langle x\rangle=0$ mean the particle is at the barrier?

<details class="lab09-answer">
<summary>Answer and explanation</summary>

Near $t=28.15$, this example has mostly transferred to the left well: about 98% of the probability is there.

The density tells us whether probability has mostly transferred or remains split. No: $\langle x\rangle=0$ does not put the particle at the barrier. An equal left/right split can have mean zero while very little probability lies near the barrier.

</details>

## How long did the first trip take?

Give the time and mean position at the first major minimum. Use the density at that time to explain whether most of the probability has reached the left well.

For your assigned $\alpha$, check that the run is long enough to see the transfer and return.

Assignment 2 is due Monday, November 2.

<details class="lab09-answer">
<summary>Answer and explanation</summary>

For this $\alpha=0.4$ example, the first broad minimum is near $t=28.15$, with $\langle x\rangle\approx-1.51$. About 98% of the probability is at $x<0$. The small ripples give several nearby local minima; read the broad transfer first.

Your assigned $\alpha$ can give a different transfer time. Extend the run if you have not yet seen the transfer and return.

</details>

Run cells in order. Shift+Enter selects the discussion question on the same slide, leaving the result visible. Click the code cell to edit and run it again; Ctrl+Enter runs without changing the selected cell. After changing `ALPHA`, rerun from the setup onward. The preview controls redraw their own plots; use `ALPHA` in the setup to change the evolution.